In [2]:
import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm



tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)




Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# try using BERT (Bidirectional Encoder Representations from Transformers)
#pre-trained on lots of text data, but we can fine-tune it to our train dataset
# essentially has "attention" mechanism to understand context of words in a sentence 
# e.g. 'Nationwide is a good way to BANK' versus 'the BANK of the river'

In [11]:
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix

In [3]:
# load data
path = "/Users/wilhelmlannin/.cache/kagglehub/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/versions/1"
df = pd.read_csv(fr"{path}/IMDB Dataset.csv")
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
# train/test split by randomly selecting 12.5k of the positives and 12.5k of the negatives (i.e. 50% of the data)

train_positive_idx = df[df['sentiment']=='positive'].sample(12500,random_state=42).index
train_negative_idx = df[df['sentiment']=='negative'].sample(12500,random_state=42).index
train_idx = list(train_positive_idx) + list(train_negative_idx)

train_df = df[df.index.isin(train_idx)]
print(train_df.shape)

test_df = df[~df.index.isin(train_idx)]
print(test_df.shape)


(25000, 2)
(25000, 2)


In [5]:
# define target column
train_df['target'] = np.where(train_df['sentiment']=='positive',1,0)
test_df['target'] = np.where(test_df['sentiment']=='positive',1,0)



/var/folders/6x/wbjj0_r10vv8b_8nn46bv_qr0000gn/T/ipykernel_22525/1488112568.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['target'] = np.where(train_df['sentiment']=='positive',1,0)
/var/folders/6x/wbjj0_r10vv8b_8nn46bv_qr0000gn/T/ipykernel_22525/1488112568.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['target'] = np.where(test_df['sentiment']=='positive',1,0)


In [ ]:
# class to prepare data for pytorch and BERT model
class ReviewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.reviews = df['review'].tolist()
        self.targets = df['target'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        text = self.reviews[idx]
        label = self.targets[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


In [7]:
train_dataset = ReviewsDataset(train_df, tokenizer)
test_dataset  = ReviewsDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=2, shuffle=False)


In [ ]:
# move model to GPU if available, else use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [9]:
optimizer = AdamW(model.parameters(), lr=5e-5)


In [10]:
#tune the BERT model on my training data
epochs = 2

model.train()

for epoch in range(epochs):
    loop = tqdm(train_loader, leave=True)
    total_loss = 0

    for batch in loop:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}/{epochs}")
        loop.set_postfix(loss=loss.item())
    
    print(f"Average training loss for epoch {epoch+1}: {total_loss/len(train_loader):.4f}")


Epoch 1/2: 100%|██████████| 12500/12500 [31:56<00:00,  6.52it/s, loss=0.674]


Average training loss for epoch 1: 0.6962


Epoch 2/2: 100%|██████████| 12500/12500 [32:05<00:00,  6.49it/s, loss=0.753]

Average training loss for epoch 2: 0.6943


In [ ]:
model.eval()

all_labels = []
all_probs = []

correct = 0
total = 0

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        # Convert logits to probabilities using softmax
        probs = torch.softmax(logits, dim=1)[:, 1]   # probability of class 1 (positive)

        # Predictions
        predictions = torch.argmax(logits, dim=1)

        # Correctness
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        # Save for AUC
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

accuracy = correct / total

# Compute AUC
auc = roc_auc_score(all_labels, all_probs)

print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC:  {auc:.4f}")


100%|██████████| 12500/12500 [06:34<00:00, 31.70it/s]


Test Accuracy: 0.5498
Test ROC-AUC:  0.5490


In [ ]:
# very poor performance!! I will try increasing num_tokens and batch_size

# increased compute

In [18]:
model2 = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# class to prepare data for pytorch and BERT model
class ReviewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.reviews = df['review'].tolist()
        self.targets = df['target'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        text = self.reviews[idx]
        label = self.targets[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


In [20]:
train_dataset = ReviewsDataset(train_df, tokenizer)
test_dataset  = ReviewsDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False)

# move model to GPU if available, else use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model2 = model2.to(device)

optimizer = AdamW(model2.parameters(), lr=5e-5)



In [21]:
#tune the BERT model on my training data
epochs = 5

model2.train()

for epoch in range(epochs):
    loop = tqdm(train_loader, leave=True)
    total_loss = 0

    for batch in loop:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}/{epochs}")
        loop.set_postfix(loss=loss.item())
    
    print(f"Average training loss for epoch {epoch+1}: {total_loss/len(train_loader):.4f}")


Epoch 1/5: 100%|██████████| 6250/6250 [30:33<00:00,  3.41it/s, loss=0.17]   


Average training loss for epoch 1: 0.3945


Epoch 2/5: 100%|██████████| 6250/6250 [28:50<00:00,  3.61it/s, loss=0.192]  


Average training loss for epoch 2: 0.2505


Epoch 3/5: 100%|██████████| 6250/6250 [30:12<00:00,  3.45it/s, loss=0.0151]  


Average training loss for epoch 3: 0.1503


Epoch 4/5: 100%|██████████| 6250/6250 [29:53<00:00,  3.49it/s, loss=0.00897] 


Average training loss for epoch 4: 0.0914


Epoch 5/5: 100%|██████████| 6250/6250 [29:52<00:00,  3.49it/s, loss=0.0529]  

Average training loss for epoch 5: 0.0608


In [22]:
model2.eval()

all_labels = []
all_probs = []

correct = 0
total = 0

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model2(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        # Convert logits to probabilities using softmax
        probs = torch.softmax(logits, dim=1)[:, 1]   # probability of class 1 (positive)

        # Predictions
        predictions = torch.argmax(logits, dim=1)

        # Correctness
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        # Save for AUC
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

accuracy = correct / total

# Compute AUC
auc = roc_auc_score(all_labels, all_probs)

print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC:  {auc:.4f}")


100%|██████████| 6250/6250 [07:00<00:00, 14.86it/s]



Test Accuracy: 0.8212
Test ROC-AUC:  0.8993


In [23]:
# eval on training data too
model2.eval()

all_labels = []
all_probs = []

correct = 0
total = 0

with torch.no_grad():
    for batch in tqdm(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model2(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        # Convert logits to probabilities using softmax
        probs = torch.softmax(logits, dim=1)[:, 1]   # probability of class 1 (positive)

        # Predictions
        predictions = torch.argmax(logits, dim=1)

        # Correctness
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        # Save for AUC
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

accuracy = correct / total

# Compute AUC
auc = roc_auc_score(all_labels, all_probs)

print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC:  {auc:.4f}")


100%|██████████| 6250/6250 [06:54<00:00, 15.08it/s]


Test Accuracy: 0.9908
Test ROC-AUC:  0.9994
